## Tool Creation Methods: Quick Summary

| # | Method | One-line concept | When to use |
|---|--------|------------------|-------------|
| 1 | `@tool` | Normal Python function ko decorator laga ke tool bana deta hai (name = function name, description = docstring) | Default choice, simple custom function ke liye |
| 2 | `@tool("custom_name")` | Same as `@tool`, bas tool ka naam apni marzi ka rakh sakte ho | Jab function name aur tool name alag chahiye |
| 3 | `@tool(name, description=..., return_direct=...)` | Decorator me hi name, description aur behavior options set karo | Jab docstring ke bajaye explicit description/options dene hon |
| 4 | `@tool(args_schema=PydanticModel)` | Pydantic model se input validate hota hai (galat type pe error aata hai) | Complex ya validated business input ke liye |
| 5 | `@tool(parse_docstring=True)` | Docstring ke `Args:` section se har argument ki description automatically banti hai | Jab arguments ki clear description LLM ko deni ho, bina Pydantic likhe |
| 6 | `async @tool` | Async function ka tool, `ainvoke` se chalta hai | API, DB, network calls jahan wait karna padta hai |
| 7 | `ToolRuntime` | Tool ko graph ka state/context/store access dene ke liye | Jab tool ko current state ya memory padhni ho |
| 8 | `Tool(...)` / `Tool.from_function()` | Simple function ko explicit wrapper me tool banata hai (single string input) | Purane/simple single-input wrappers ke liye |
| 9 | `StructuredTool.from_function()` / `StructuredTool(...)` | Multiple typed inputs wala tool explicitly construct karta hai | Jab decorator nahi use karna, programmatically tool banana ho |
| 10 | Subclass `BaseTool` | Class likh ke `_run` method me poori custom logic dete ho | Advanced customization, state/config wali class-based tools |
| 11 | `create_retriever_tool()` | Retriever (vector DB search) ko seedha tool bana deta hai | Agentic RAG, jahan LLM decide kare ke documents search karne hain ya nahi |
| 12 | Toolkit (`BaseToolkit`) | Related tools ka ek group/collection return karta hai | Jab ek hi domain ke kai tools saath manage karne hon |
| 13 | Prebuilt integration (e.g. `TavilySearch`) | Ready-made tool, bas import karo aur use karo | Web search, DB, APIs jinke liye tool pehle se bana hua hai |
| 14 | MCP tools (`FastMCP` + `MultiServerMCPClient`) | Alag MCP server pe bane tools ko LangChain tools ki tarah load karta hai | Shared ya remote tools, jo kai applications use karein |
| 15 | `to_fastmcp()` | LangChain tool ko MCP server ke tool me convert karta hai | Apna LangChain tool dusre MCP clients ko dena ho |
| 16 | `runnable.as_tool()` | Existing chain/Runnable ko tool bana deta hai | Jab LCEL chain pehle se bani ho aur tool banani ho |
| 17 | Pydantic / TypedDict / raw JSON schema + `bind_tools()` | Sirf schema deta hai, function nahi (LLM sirf tool call generate karta hai) | Low-level/provider-level kaam, ya jab execution alag handle karna ho |

### Creation methods nahi hain (yeh tools ko *use* karte hain)

| Name | One-line concept | When to use |
|------|------------------|-------------|
| `bind_tools()` | Tools ko LLM ke saath attach karta hai taake woh `tool_calls` generate kar sake | Tool banane ke baad, LLM ko batane ke liye |
| `ToolNode` | LangGraph node jo LLM ke `tool_calls` ko actually execute karta hai | Agent graph me tool execution ke liye |

Tool calling means an LLM can decide to use an external function/tool when it needs information or wants to perform an action, instead of answering only from its own knowledge.

Tool calling is the ability of an LLM to select and invoke external tools or functions to get information or perform actions beyond its built-in knowledge.

## these are some of the example 
### but the concept is we can convert any of the funcationality into the tool

Web search
Calculator
Database query
API calls
retriever
Email sending
Weather API
Stock-price API
File operations

User: "What is the weather in Bangalore today?"

        ↓

LLM decides: "I need current weather data."

        ↓

Calls Weather Tool

        ↓

Tool returns: 28°C, cloudy

        ↓

LLM gives final answer

Node = graph decides when to execute the function.
Tool = LLM decides when to execute the function.

User
 ↓
LLM
 ↓
Does it need a tool?
 ↓
Yes
 ↓
Tool(function) Call
 ↓
Tool Result
 ↓
LLM
 ↓
Final Answer

```
MCP Server
   │
   ├── Tool 1
   ├── Tool 2
   └── Tool 3
        ↓
langchain-mcp-adapters
        ↓
LangChain BaseTools
        ↓
LangGraph Agent

```

Now if the user asks: "What is 20 + 30?"

the LLM may produce a tool request like:

Tool: add
Arguments:
a = 20
b = 30

Then the tool executes: 50

and the LLM can use that result to answer.

The LLM does not execute the function itself. It decides which tool(function) to call and with what arguments; your application executes the tool and returns the result to the LLM.

LLM = decision maker
Tool = actual executor(acutal funcationality)

## 1. `@tool` decorator

In [62]:
from langchain.tools import tool 

@tool()
def add_basis(a: int, b: int) -> int:
    "add two number"
    return a+b 

print(f"Name : {add_basis.name}")
print(f"Description : {add_basis.description}")
print(f"args : {add_basis.args}")
print(f"run : {add_basis.invoke({"a": 20 , "b": 30})}")

Name : add_basis
Description : add two number
args : {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
run : 50


## 2. `@tool("custom_name")`

In [63]:
@tool("calculator")
def add_with_custom_name(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

print("Tool name:", add_with_custom_name.name)
print("Execution result:", add_with_custom_name.invoke({"a": 10, "b": 15}))

Tool name: calculator
Execution result: 25


In [64]:
@tool(
    "multiply_numbers",
    description="Multiply two integers and return the result.",
    return_direct=False,
)
def multiply_with_options(a, b):
    return a * b

print("Name:", multiply_with_options.name)
print("Description:", multiply_with_options.description)
print("return_direct:", multiply_with_options.return_direct)
print("Execution result:", multiply_with_options.invoke({"a": 6, "b": 7}))

Name: multiply_numbers
Description: Multiply two integers and return the result.
return_direct: False
Execution result: 42


## 4. `@tool` with Pydantic

In [65]:
from pydantic import BaseModel, Field, ValidationError

class CalculatorInputTest(BaseModel):
    a: int = Field(description="First integer")
    b: int = Field(description="Second integer")

@tool(args_schema=CalculatorInputTest)
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b
print("Execution result:", multiply.invoke({"a": 8, "b": 9}))

Execution result: 72


In [66]:
print("Schema:", multiply.args_schema.model_json_schema())

Schema: {'properties': {'a': {'description': 'First integer', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second integer', 'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'CalculatorInputTest', 'type': 'object'}


## 5. `@tool(parse_docstring=True)`

In [67]:
@tool(parse_docstring=True)
def search_with_docstring(query: str, limit: int) -> str:
    """Search documents.

    Args:
        query: Search query entered by the user.
        limit: Maximum number of results.
    """
    return f"Searching for '{query}' with limit={limit}"
print("Execution result:", search_with_docstring.invoke({"query": "LangGraph memory","limit": 3,}))


Execution result: Searching for 'LangGraph memory' with limit=3


In [68]:
print("Args schema:")
print(search_with_docstring.args_schema.model_json_schema())

Args schema:
{'description': 'Search documents.', 'properties': {'query': {'description': 'Search query entered by the user.', 'title': 'Query', 'type': 'string'}, 'limit': {'description': 'Maximum number of results.', 'title': 'Limit', 'type': 'integer'}}, 'required': ['query', 'limit'], 'title': 'search_with_docstring', 'type': 'object'}


## 6. Async function with `@tool`

In [69]:
import asyncio

In [70]:
@tool
async def get_data_async(url: str) -> str:
    """Fetch data asynchronously."""
    await asyncio.sleep(0.1)
    return f"Data from {url}"

result = await get_data_async.ainvoke({"url": "https://example.com"})
print("Async execution result:", result)

Async execution result: Data from https://example.com


## 7. Tool with `ToolRuntime` 

<!-- from typing_extensions import TypedDict
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, ToolRuntime
from langchain_core.messages import AIMessage

class RuntimeState(MessagesState):
    question: str

@tool
def read_question_from_runtime(runtime: ToolRuntime) -> str:
    """Read the current question from LangGraph state."""
    return runtime.state["question"]

def create_demo_tool_call(state: RuntimeState):
    return {
        "messages": [
            AIMessage(
                content="",
                tool_calls=[{
                    "name": "read_question_from_runtime",
                    "args": {},
                    "id": "demo_call_1",
                    "type": "tool_call",
                }],
            )
        ]
    }

runtime_builder = StateGraph(RuntimeState)
runtime_builder.add_node("create_call", create_demo_tool_call)
runtime_builder.add_node("tools", ToolNode([read_question_from_runtime]))
runtime_builder.add_edge(START, "create_call")
runtime_builder.add_edge("create_call", "tools")
runtime_builder.add_edge("tools", END)

runtime_graph = runtime_builder.compile()

runtime_result = runtime_graph.invoke({
    "question": "What is LangGraph?",
    "messages": [],
})

print("Tool result:", runtime_result["messages"][-1].content) -->

## 8. `Tool(...)` constructor"

In [71]:
from langchain_core.tools import Tool

def simple_search_function(query: str) -> str:
    return f"Searching for {query}"

simple_search_tool = Tool(
    name="simple_search",
    func=simple_search_function,
    description="Search for information.",
)

print("Name:", simple_search_tool.name)
print("Execution result:", simple_search_tool.invoke("LangGraph"))

Name: simple_search
Execution result: Searching for LangGraph


In [72]:
def search_from_function(query: str) -> str:
    return f"Result for {query}"

from_function_tool = Tool.from_function(
    func=search_from_function,
    name="search_from_function",
    description="Search information.",
)
print("Execution result:", from_function_tool.invoke("Agentic AI"))

Execution result: Result for Agentic AI


## 10. `StructuredTool.from_function()`

In [73]:
from langchain_core.tools import StructuredTool

def calculate_tax_test(income: float, tax_rate: float) -> float:
    return income * tax_rate

tax_tool_test = StructuredTool.from_function(
    func=calculate_tax_test,
    name="calculate_tax",
    description="Calculate tax from income and tax rate.",
)

print("Args:", tax_tool_test.args)
print("Execution result:", tax_tool_test.invoke({
    "income": 100000,
    "tax_rate": 0.20,
}))

Args: {'income': {'title': 'Income', 'type': 'number'}, 'tax_rate': {'title': 'Tax Rate', 'type': 'number'}}
Execution result: 20000.0


In [74]:
class MultiplyInputTest2(BaseModel):
    a: int = Field(description="First number")
    b: int = Field(description="Second number")

def direct_multiply(a: int, b: int) -> int:
    return a * b

direct_structured_tool = StructuredTool(
    name="direct_multiply",
    description="Multiply two numbers.",
    func=direct_multiply,
    args_schema=MultiplyInputTest2,
)

print("Execution result:", direct_structured_tool.invoke({
    "a": 12,
    "b": 4,
}))

Execution result: 48


## 12. Subclass `BaseTool`

In [75]:
from typing import Type
from langchain_core.tools import BaseTool

class SearchInputTest(BaseModel):
    query: str = Field(description="Search query")
    

class MySearchToolTest(BaseTool):
    name: str = "my_search"
    description: str = "Search my custom database."
    args_schema: Type[BaseModel] = SearchInputTest

    def _run(self, query: str) -> str:
        return f"Custom database result for: {query}"

custom_base_tool = MySearchToolTest()
print("Execution result:", custom_base_tool.invoke({
    "query": "LangGraph state management"
}))

Execution result: Custom database result for: LangGraph state management


## 13. `create_retriever_tool()`

In [76]:
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.tools import create_retriever_tool

In [77]:
class DemoRetriever(BaseRetriever):
    def _get_relevant_documents(self, query: str, *, run_manager=None):
        return [
            Document(
                page_content=f"Demo internal document relevant to: {query}",
                metadata={"source": "demo.txt"},
            )
        ]

demo_retriever = DemoRetriever()

In [78]:
retriever_tool_test = create_retriever_tool(
    demo_retriever,
    name="search_company_documents",
    description="Search internal company documents.",
)

print("Execution result:")
print(retriever_tool_test.invoke({
    "query": "What is the company leave policy?"
}))

Execution result:
Demo internal document relevant to: What is the company leave policy?


## Model setup for schema/binding tests

In [79]:
import os 
from langchain_groq import ChatGroq


groq_llm = ChatGroq(model="openai/gpt-oss-20b",temperature=0,api_key=os.getenv("GROQ_API_KEY"))

@tool
def weather_callable(location: str) -> str:
    """Get weather for a location."""
    return f"Demo weather for {location}: 28°C"

callable_bound_model = groq_llm.bind_tools([weather_callable])
ai_message = callable_bound_model.invoke("Use the weather tool for Bangalore.")
ai_message.tool_calls

[10/13/26 15:32:15] INFO     HTTP Request: POST https://api.groq.com/openai/v1/chat/completions     ]8;id=12883086;file://c:\Users\abubakar\miniconda3\envs\genai\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=12883087;file://c:\Users\abubakar\miniconda3\envs\genai\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[{'name': 'weather_callable',
  'args': {'location': 'Bangalore'},
  'id': 'fc_39ef9375-b143-46ab-a3dc-efdbb5d2899a',
  'type': 'tool_call'}]

##### Toolkit — a collection/factory of related tools

In [80]:
from langchain_core.tools import BaseToolkit

@tool
def toolkit_add(a: int, b: int) -> int:
    """Add numbers."""
    return a + b

@tool
def toolkit_multiply(a: int, b: int) -> int:
    """Multiply numbers."""
    return a * b

class DemoMathToolkit(BaseToolkit):
    def get_tools(self):
        return [
            toolkit_add,
            toolkit_multiply,
        ]

demo_toolkit = DemoMathToolkit()
toolkit_tools = demo_toolkit.get_tools()

print("Toolkit tools:", [t.name for t in toolkit_tools])
print("Add result:", toolkit_tools[0].invoke({"a": 2, "b": 3}))
print("Multiply result:", toolkit_tools[1].invoke({"a": 4, "b": 5}))

Toolkit tools: ['toolkit_add', 'toolkit_multiply']
Add result: 5
Multiply result: 20


#### Prebuilt integration tool

In [81]:
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(
    max_results=3,
    search_depth="basic",
    topic="general",
    api_key = os.getenv("TAVILY_API_KEY")
)

result = tavily_tool.invoke({
    "query": "What is LangGraph?"
})

first_result = result.get("results", [{}])[0]
print(first_result.get("content", "No content returned."))

LangGraph is an open-source framework from LangChain designed to build and manage AI agent workflows using graph-based structures. It allows developers to define workflows as nodes and edges, making complex agent interactions more structured, scalable and easier to control. [...] langgraph: Framework for building graph-based AI workflows.
 langchain: Popular toolkit for LLM-powered AI applications.
 google-generativeai: Google’s API for Generative AI (Gemini models).

 Python  ````
! pip install langgraph langchain google - generativeai
```` 

### Step 2: Setup Gemini API


### Note

 `bind_tools()` and `ToolNode` are NOT creation methods

 ```text
Create Tool
    ↓
bind_tools()
    ↓
LLM generates tool_calls
    ↓
ToolNode executes the call
    ↓
ToolMessage
    ↓
LLM can continue
```

#### Create an MCP tool and load it into LangChain

```bash
uv pip install -U mcp langchain-mcp-adapters
```

| Priority | Method                           | When                                 |
| -------- | -------------------------------- | ------------------------------------ |
| ⭐⭐⭐⭐⭐    | `@tool`                          | Default for normal custom functions  |
| ⭐⭐⭐⭐⭐    | `@tool + Pydantic`               | Complex/validated business input     |
| ⭐⭐⭐⭐     | async `@tool`                    | API/DB/network tools                 |
| ⭐⭐⭐⭐     | `create_retriever_tool()`        | Agentic RAG                          |
| ⭐⭐⭐⭐     | Prebuilt integrations            | Tavily, DB, APIs, etc.               |
| ⭐⭐⭐⭐     | MCP                              | External/shared enterprise tools     |
| ⭐⭐⭐      | `ToolRuntime`                    | Tool needs graph state/context/store |
| ⭐⭐⭐      | Runnable `.as_tool()`            | Existing chain/Runnable becomes tool |
| ⭐⭐       | `StructuredTool.from_function()` | Explicit structured construction     |
| ⭐⭐       | `BaseTool`                       | Advanced custom integration          |
| ⭐        | `Tool(...)` / `.from_function()` | Simple explicit wrappers             |
| ⭐        | Raw JSON schemas                 | Provider-level/low-level work        |


My decision tree

Use this in your class:
```
I have a normal Python function
        |
        v
      @tool
        |
        +--------------------------+
        |                          |
Simple inputs?               Complex inputs?
        |                          |
       YES                         YES
        |                          |
      @tool              @tool + Pydantic

Then:

Does it call network/API/DB?
        |
       YES
        |
        v
   async @tool

Then:

Is it a Retriever?
        |
       YES
        |
        v
create_retriever_tool()

Then:

Already have an LCEL/Runnable?
        |
       YES
        |
        v
   runnable.as_tool()

Then:

Need graph state/context/store?
        |
       YES
        |
        v
     ToolRuntime

Then:

Need very deep customization?
        |
       YES
        |
        v
     BaseTool

Then:

Tool exists remotely / shared
across many applications?
        |
       YES
        |
        v
        MCP

```

Node or Tool? Final decision tree

This is the most useful one.

Ask:

Should my workflow guarantee WHEN this function runs?

If yes:

NODE

Example:

Validate every response.

Graph:

LLM
 ↓
Validator
 ↓
END

Ask:

Should the LLM decide WHETHER it needs this capability?

If yes:

TOOL

Example:

Maybe use calculator.
Maybe use search.
Maybe don't use anything.

Flow:

LLM
 ↓
Tool needed?
 /       \
Yes       No
 ↓         ↓
Tool     Answer